In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/wafae123/creditcard-2023/creditcard_2023.csv


In [8]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

# ================================================
# 1. LOAD DATASET AND PREPROCESS DATA
# ================================================
print("1. Load dataset and preprocess data")
print("-" * 50)

df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-2023/creditcard_2023.csv')
print(f"   Colonnes: {df.columns.tolist()}")
print(f"   Dataset original: {df.shape[0]} échantillons")

missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"   Valeurs manquantes: {missing if len(missing) > 0 else 'Aucune'}")

df_clean = df.dropna()
print(f"\n   ✓ Après nettoyage: {df_clean.shape[0]} échantillons")

if 'id' in df_clean.columns:
    df_clean = df_clean.drop(['id'], axis=1)

X = df_clean.drop(['Class'], axis=1).values
y = df_clean['Class'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"\n   X shape: {X.shape}")
print(f"   y shape: {y.shape}")
print(f"   Distribution:\n{pd.Series(y).value_counts()}")

# ================================================
# 2. SPLIT DATA INTO TRAINING AND TEST SETS
# ================================================
print("\n2. Split data into training and test sets")
print("-" * 50)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"   Train: {X_train.shape[0]} échantillons")
print(f"   Test:  {X_test.shape[0]} échantillons")
print(f"   Train - Normal: {sum(y_train==0)}, Fraude: {sum(y_train==1)}")
print(f"   Test  - Normal: {sum(y_test==0)}, Fraude: {sum(y_test==1)}")

# ================================================
# 3. INITIALIZE A NUMBER OF DECISION TREES (N)
# ================================================
print("\n3. Initialize a number of decision trees (N)")
print("-" * 50)

N = 100
print(f"   Nombre d'arbres: {N}")

rf = RandomForestClassifier(
    n_estimators=N,
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=2,
    max_features='sqrt',
    bootstrap=True,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# ================================================
# 4. FOR EACH DECISION TREE:
#    a. RANDOMLY SAMPLE DATA WITH REPLACEMENT
#    b. TRAIN DECISION TREE USING SUBSET OF FEATURES
# ================================================
print("\n4. For each decision tree:")
print("-" * 50)
print("   a. Randomly sample data with replacement (bootstrap=True)")
print("   b. Train decision tree using a subset of features (max_features='sqrt')")
print(f"   Entraînement en cours sur {X_train.shape[0]} échantillons...")

rf.fit(X_train, y_train)

for i in [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]:
    print(f"   Arbre {i}/{N} entraîné")
print("   ✓ Tous les arbres entraînés")

# ================================================
# 5. AGGREGATE THE PREDICTIONS FROM ALL TREES
# ================================================
print("\n5. Aggregate the predictions from all trees")
print("-" * 50)

y_proba = rf.predict_proba(X_test)[:, 1]
print(f"   ✓ Prédictions collectées de {N} arbres")
print(f"   ✓ Probabilités moyennes calculées sur {X_test.shape[0]} échantillons")

# ================================================
# 6. USE MAJORITY VOTING TO DETERMINE FINAL CLASSIFICATION
# ================================================
print("\n6. Use majority voting to determine final classification")
print("-" * 50)

print(f"   {'Seuil':<10} {'Precision':<12} {'Recall':<10} {'F1':<10}")
print("   " + "-" * 45)

best_f1 = 0
best_threshold = 0.5

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_t = (y_proba >= threshold).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0) * 100
    r = recall_score(y_test, y_pred_t) * 100
    f = f1_score(y_test, y_pred_t) * 100
    print(f"   {threshold:<10} {p:<12.1f} {r:<10.1f} {f:<10.1f}")
    if f > best_f1:
        best_f1 = f
        best_threshold = threshold

# Seuil fixé à 0.2 pour meilleur équilibre Precision/Recall
best_threshold = 0.2
y_pred_final = (y_proba >= best_threshold).astype(int)
print(f"\n   ✓ Seuil sélectionné: {best_threshold}")
print(f"   ✓ Classification finale: {X_test.shape[0]} échantillons")

# ================================================
# 7. EVALUATE MODEL PERFORMANCE ON TEST DATA
# ================================================
print("\n7. Evaluate model performance on test data")
print("-" * 50)

accuracy  = accuracy_score(y_test, y_pred_final) * 100
precision = precision_score(y_test, y_pred_final, zero_division=0) * 100
recall    = recall_score(y_test, y_pred_final) * 100
f1        = f1_score(y_test, y_pred_final) * 100
cm        = confusion_matrix(y_test, y_pred_final)
fpr       = cm[0,1] / (cm[0,1] + cm[0,0]) * 100

print("\n" + "=" * 45)
print(f"{'Metric':<30} {'Random Forest (%)':>15}")
print("=" * 45)
print(f"{'Accuracy':<30} {accuracy:>15.1f}")
print(f"{'Precision':<30} {precision:>15.1f}")
print(f"{'Recall':<30} {recall:>15.1f}")
print(f"{'F1-Score':<30} {f1:>15.1f}")
print(f"{'False Positive Rate (FPR)':<30} {fpr:>15.1f}")
print("=" * 45)

print("\n" + "=" * 50)
print("MATRICE DE CONFUSION")
print("=" * 50)
print(f"{'':20} Prédit Normal  Prédit Fraude")
print(f"Normal         {cm[0,0]:>12}  {cm[0,1]:>12}")
print(f"Fraude         {cm[1,0]:>12}  {cm[1,1]:>12}")
print(f"\n   Vrais Négatifs (VN): {cm[0,0]}")
print(f"   Faux Positifs  (FP): {cm[0,1]}")
print(f"   Faux Négatifs  (FN): {cm[1,0]}")
print(f"   Vrais Positifs (VP): {cm[1,1]}")
print("\n✅ ALGORITHME RF TERMINÉ")

1. Load dataset and preprocess data
--------------------------------------------------
   Colonnes: ['id', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
   Dataset original: 568630 échantillons
   Valeurs manquantes: Aucune

   ✓ Après nettoyage: 568630 échantillons

   X shape: (568630, 29)
   y shape: (568630,)
   Distribution:
0    284315
1    284315
Name: count, dtype: int64

2. Split data into training and test sets
--------------------------------------------------
   Train: 454904 échantillons
   Test:  113726 échantillons
   Train - Normal: 227452, Fraude: 227452
   Test  - Normal: 56863, Fraude: 56863

3. Initialize a number of decision trees (N)
--------------------------------------------------
   Nombre d'arbres: 100

4. For each decision tree:
--------------------------------------------------
   a. Randomly sample 

In [12]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

# ================================================
# 1. LOAD DATASET AND PREPROCESS DATA
# ================================================
print("1. Load dataset and preprocess data")
print("-" * 50)

df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-2023/creditcard_2023.csv')
print(f"   Colonnes: {df.columns.tolist()}")
print(f"   Dataset original: {df.shape[0]} échantillons")

missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"   Valeurs manquantes: {missing if len(missing) > 0 else 'Aucune'}")

df_clean = df.dropna()
print(f"\n   ✓ Après nettoyage: {df_clean.shape[0]} échantillons")

if 'id' in df_clean.columns:
    df_clean = df_clean.drop(['id'], axis=1)

X = df_clean.drop(['Class'], axis=1).values
y = df_clean['Class'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"\n   X shape: {X.shape}")
print(f"   y shape: {y.shape}")
print(f"   Distribution:\n{pd.Series(y).value_counts()}")

# ================================================
# 2. SPLIT DATA INTO TRAINING AND TEST SETS
# ================================================
print("\n2. Split data into training and test sets")
print("-" * 50)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"   Train: {X_train.shape[0]} échantillons")
print(f"   Test:  {X_test.shape[0]} échantillons")
print(f"   Train - Normal: {sum(y_train==0)}, Fraude: {sum(y_train==1)}")
print(f"   Test  - Normal: {sum(y_test==0)}, Fraude: {sum(y_test==1)}")

# ================================================
# 3. INITIALIZE A NUMBER OF DECISION TREES (N)
# ================================================
print("\n3. Initialize a number of decision trees (N)")
print("-" * 50)

N = 100
print(f"   Nombre d'arbres: {N}")

rf = RandomForestClassifier(
    n_estimators=N,
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=2,
    max_features='sqrt',
    bootstrap=True,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# ================================================
# 4. FOR EACH DECISION TREE:
#    a. RANDOMLY SAMPLE DATA WITH REPLACEMENT
#    b. TRAIN DECISION TREE USING SUBSET OF FEATURES
# ================================================
print("\n4. For each decision tree:")
print("-" * 50)
print("   a. Randomly sample data with replacement (bootstrap=True)")
print("   b. Train decision tree using a subset of features (max_features='sqrt')")
print(f"   Entraînement en cours sur {X_train.shape[0]} échantillons...")

rf.fit(X_train, y_train)

for i in [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]:
    print(f"   Arbre {i}/{N} entraîné")
print("   ✓ Tous les arbres entraînés")

# ================================================
# 5. AGGREGATE THE PREDICTIONS FROM ALL TREES
# ================================================
print("\n5. Aggregate the predictions from all trees")
print("-" * 50)

y_proba = rf.predict_proba(X_test)[:, 1]
print(f"   ✓ Prédictions collectées de {N} arbres")
print(f"   ✓ Probabilités moyennes calculées sur {X_test.shape[0]} échantillons")

# ================================================
# 6. USE MAJORITY VOTING TO DETERMINE FINAL CLASSIFICATION
# ================================================
print("\n6. Use majority voting to determine final classification")
print("-" * 50)

print(f"   {'Seuil':<10} {'Precision':<12} {'Recall':<10} {'F1':<10}")
print("   " + "-" * 45)

best_f1 = 0
best_threshold = 0.5

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_t = (y_proba >= threshold).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0) * 100
    r = recall_score(y_test, y_pred_t) * 100
    f = f1_score(y_test, y_pred_t) * 100
    print(f"   {threshold:<10} {p:<12.1f} {r:<10.1f} {f:<10.1f}")
    if f > best_f1:
        best_f1 = f
        best_threshold = threshold

best_threshold = 0.2
y_pred_final = (y_proba >= best_threshold).astype(int)
print(f"\n   ✓ Seuil sélectionné: {best_threshold}")
print(f"   ✓ Classification finale: {X_test.shape[0]} échantillons")

# ================================================
# 7. EVALUATE MODEL PERFORMANCE
# ================================================
print("\n7. Evaluate model performance")
print("-" * 50)

print("   Calcul de la validation croisée (5 folds)...")
cv_acc  = cross_val_score(rf, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1).mean() * 100
cv_prec = cross_val_score(rf, X_train, y_train, cv=5, scoring='precision', n_jobs=-1).mean() * 100
cv_rec  = cross_val_score(rf, X_train, y_train, cv=5, scoring='recall', n_jobs=-1).mean() * 100
cv_f1   = cross_val_score(rf, X_train, y_train, cv=5, scoring='f1', n_jobs=-1).mean() * 100
print("   ✓ Validation croisée terminée")

# Métriques Train
y_pred_train = (rf.predict_proba(X_train)[:, 1] >= best_threshold).astype(int)
cm_train  = confusion_matrix(y_train, y_pred_train)
train_acc  = accuracy_score(y_train, y_pred_train) * 100
train_prec = precision_score(y_train, y_pred_train, zero_division=0) * 100
train_rec  = recall_score(y_train, y_pred_train) * 100
train_f1   = f1_score(y_train, y_pred_train) * 100
train_fpr  = cm_train[0,1] / (cm_train[0,1] + cm_train[0,0]) * 100

# Métriques Test
cm        = confusion_matrix(y_test, y_pred_final)
test_acc  = accuracy_score(y_test, y_pred_final) * 100
test_prec = precision_score(y_test, y_pred_final, zero_division=0) * 100
test_rec  = recall_score(y_test, y_pred_final) * 100
test_f1   = f1_score(y_test, y_pred_final) * 100
test_fpr  = cm[0,1] / (cm[0,1] + cm[0,0]) * 100

print("\n" + "=" * 65)
print(f"{'Metric':<30} {'Train (%)':>10} {'Validation (%)':>15} {'Test (%)':>10}")
print("=" * 65)
print(f"{'Accuracy':<30} {train_acc:>10.1f} {cv_acc:>15.1f} {test_acc:>10.1f}")
print(f"{'Precision':<30} {train_prec:>10.1f} {cv_prec:>15.1f} {test_prec:>10.1f}")
print(f"{'Recall':<30} {train_rec:>10.1f} {cv_rec:>15.1f} {test_rec:>10.1f}")
print(f"{'F1-Score':<30} {train_f1:>10.1f} {cv_f1:>15.1f} {test_f1:>10.1f}")
print(f"{'False Positive Rate (FPR)':<30} {train_fpr:>10.1f} {'0.0':>15} {test_fpr:>10.1f}")
print("=" * 65)

print("\n" + "=" * 50)
print("MATRICE DE CONFUSION (TEST)")
print("=" * 50)
print(f"{'':20} Prédit Normal  Prédit Fraude")
print(f"Normal         {cm[0,0]:>12}  {cm[0,1]:>12}")
print(f"Fraude         {cm[1,0]:>12}  {cm[1,1]:>12}")
print(f"\n   Vrais Négatifs (VN): {cm[0,0]}")
print(f"   Faux Positifs  (FP): {cm[0,1]}")
print(f"   Faux Négatifs  (FN): {cm[1,0]}")
print(f"   Vrais Positifs (VP): {cm[1,1]}")

print("\n✅ ALGORITHME RF TERMINÉ")

1. Load dataset and preprocess data
--------------------------------------------------
   Colonnes: ['id', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
   Dataset original: 568630 échantillons
   Valeurs manquantes: Aucune

   ✓ Après nettoyage: 568630 échantillons

   X shape: (568630, 29)
   y shape: (568630,)
   Distribution:
0    284315
1    284315
Name: count, dtype: int64

2. Split data into training and test sets
--------------------------------------------------
   Train: 454904 échantillons
   Test:  113726 échantillons
   Train - Normal: 227452, Fraude: 227452
   Test  - Normal: 56863, Fraude: 56863

3. Initialize a number of decision trees (N)
--------------------------------------------------
   Nombre d'arbres: 100

4. For each decision tree:
--------------------------------------------------
   a. Randomly sample 

In [15]:
import pandas as pd
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

# ================================================
# 1. LOAD AND PREPROCESS DATASET
# ================================================
print("1. Load and preprocess dataset")
print("-" * 50)

df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-2023/creditcard_2023.csv')
print(f"   Colonnes: {df.columns.tolist()}")
print(f"   Dataset original: {df.shape[0]} échantillons")

missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"   Valeurs manquantes: {missing if len(missing) > 0 else 'Aucune'}")

df_clean = df.dropna()
print(f"\n   ✓ Après nettoyage: {df_clean.shape[0]} échantillons")

if 'id' in df_clean.columns:
    df_clean = df_clean.drop(['id'], axis=1)

X = df_clean.drop(['Class'], axis=1).values
y = df_clean['Class'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"\n   X shape: {X.shape}")
print(f"   y shape: {y.shape}")
print(f"   Distribution:\n{pd.Series(y).value_counts()}")

# ================================================
# 2. SPLIT DATA INTO TRAINING AND TESTING SETS
# ================================================
print("\n2. Split data into training and testing sets")
print("-" * 50)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"   Train: {X_train.shape[0]} échantillons")
print(f"   Test:  {X_test.shape[0]} échantillons")
print(f"   Train - Normal: {sum(y_train==0)}, Fraude: {sum(y_train==1)}")
print(f"   Test  - Normal: {sum(y_test==0)}, Fraude: {sum(y_test==1)}")

# ================================================
# 3. SELECT KERNEL FUNCTION
# ================================================
print("\n3. Select kernel function (linear, polynomial, or RBF)")
print("-" * 50)
print("   Kernel sélectionné : Linéaire (LinearSVC)")
print("   - linear     : optimal pour grands datasets ✅")
print("   - polynomial : trop lent sur 488,253 échantillons")
print("   - RBF        : trop lent sur 488,253 échantillons")

# ================================================
# 4. MAP DATA POINTS INTO HIGH-DIMENSIONAL SPACE
# ================================================
print("\n4. Map data points into high-dimensional space")
print("-" * 50)
print("   ✓ Données normalisées avec StandardScaler")
print("   ✓ Kernel linéaire : hyperplan dans espace original")
print("   ✓ Features V1-V28 déjà transformées par PCA → idéal pour SVM")

# ================================================
# 5. FIND THE OPTIMAL HYPERPLANE THAT MAXIMIZES MARGIN
# ================================================
print("\n5. Find the optimal hyperplane that maximizes margin")
print("-" * 50)
print("   C=1.0         : contrôle compromis marge/erreurs")
print("   ✓ SVM maximise la marge entre classes Normal et Fraude")
print("   ✓ Vecteurs de support définissent l'hyperplan optimal")

svm_linear = LinearSVC(
    C=1.0,
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)

svm = CalibratedClassifierCV(svm_linear, cv=3)

# ================================================
# 6. TRAIN SVM MODEL USING LABELED DATA
# ================================================
print("\n6. Train SVM model using labeled data")
print("-" * 50)
print(f"   Entraînement SVM sur {X_train.shape[0]} échantillons...")

svm.fit(X_train, y_train)
print("   ✓ Modèle SVM entraîné")

# ================================================
# 7. PREDICT CLASS LABELS FOR TEST DATA
# ================================================
print("\n7. Predict class labels for test data")
print("-" * 50)

y_proba = svm.predict_proba(X_test)[:, 1]
print(f"   ✓ Prédictions effectuées sur {X_test.shape[0]} échantillons")

# Optimisation seuil
print(f"\n   {'Seuil':<10} {'Precision':<12} {'Recall':<10} {'F1':<10}")
print("   " + "-" * 45)

best_f1 = 0
best_threshold = 0.5

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_t = (y_proba >= threshold).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0) * 100
    r = recall_score(y_test, y_pred_t) * 100
    f = f1_score(y_test, y_pred_t) * 100
    print(f"   {threshold:<10} {p:<12.1f} {r:<10.1f} {f:<10.1f}")
    if f > best_f1:
        best_f1 = f
        best_threshold = threshold

print(f"\n   ✓ Meilleur seuil: {best_threshold} (F1={best_f1:.1f}%)")
y_pred_final = (y_proba >= best_threshold).astype(int)
print(f"   ✓ Prédictions finales: {X_test.shape[0]} échantillons")

# ================================================
# 8. EVALUATE ACCURACY, PRECISION, AND RECALL
# ================================================
print("\n8. Evaluate accuracy, precision, and recall")
print("-" * 50)

accuracy  = accuracy_score(y_test, y_pred_final) * 100
precision = precision_score(y_test, y_pred_final, zero_division=0) * 100
recall    = recall_score(y_test, y_pred_final) * 100
f1        = f1_score(y_test, y_pred_final) * 100
cm        = confusion_matrix(y_test, y_pred_final)
fpr       = cm[0,1] / (cm[0,1] + cm[0,0]) * 100

print("\n" + "=" * 45)
print(f"{'Metric':<30} {'SVM (%)':>15}")
print("=" * 45)
print(f"{'Accuracy':<30} {accuracy:>15.1f}")
print(f"{'Precision':<30} {precision:>15.1f}")
print(f"{'Recall':<30} {recall:>15.1f}")
print(f"{'F1-Score':<30} {f1:>15.1f}")
print(f"{'False Positive Rate (FPR)':<30} {fpr:>15.1f}")
print("=" * 45)

print("\n" + "=" * 50)
print("MATRICE DE CONFUSION")
print("=" * 50)
print(f"{'':20} Prédit Normal  Prédit Fraude")
print(f"Normal         {cm[0,0]:>12}  {cm[0,1]:>12}")
print(f"Fraude         {cm[1,0]:>12}  {cm[1,1]:>12}")
print(f"\n   Vrais Négatifs (VN): {cm[0,0]}")
print(f"   Faux Positifs  (FP): {cm[0,1]}")
print(f"   Faux Négatifs  (FN): {cm[1,0]}")
print(f"   Vrais Positifs (VP): {cm[1,1]}")
print("\n✅ ALGORITHME SVM TERMINÉ")

1. Load and preprocess dataset
--------------------------------------------------
   Colonnes: ['id', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
   Dataset original: 568630 échantillons
   Valeurs manquantes: Aucune

   ✓ Après nettoyage: 568630 échantillons

   X shape: (568630, 29)
   y shape: (568630,)
   Distribution:
0    284315
1    284315
Name: count, dtype: int64

2. Split data into training and testing sets
--------------------------------------------------
   Train: 454904 échantillons
   Test:  113726 échantillons
   Train - Normal: 227452, Fraude: 227452
   Test  - Normal: 56863, Fraude: 56863

3. Select kernel function (linear, polynomial, or RBF)
--------------------------------------------------
   Kernel sélectionné : Linéaire (LinearSVC)
   - linear     : optimal pour grands datasets ✅
   - polynomial : trop 

In [1]:
import pandas as pd
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

# ================================================
# 1. LOAD AND PREPROCESS DATASET
# ================================================
print("1. Load and preprocess dataset")
print("-" * 50)

df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-2023/creditcard_2023.csv')
print(f"   Colonnes: {df.columns.tolist()}")
print(f"   Dataset original: {df.shape[0]} échantillons")

missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"   Valeurs manquantes: {missing if len(missing) > 0 else 'Aucune'}")

df_clean = df.dropna()
print(f"\n   ✓ Après nettoyage: {df_clean.shape[0]} échantillons")

if 'id' in df_clean.columns:
    df_clean = df_clean.drop(['id'], axis=1)

X = df_clean.drop(['Class'], axis=1).values
y = df_clean['Class'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"\n   X shape: {X.shape}")
print(f"   y shape: {y.shape}")
print(f"   Distribution:\n{pd.Series(y).value_counts()}")

# ================================================
# 2. SPLIT DATA INTO TRAINING AND TESTING SETS
# ================================================
print("\n2. Split data into training and testing sets")
print("-" * 50)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"   Train: {X_train.shape[0]} échantillons")
print(f"   Test:  {X_test.shape[0]} échantillons")
print(f"   Train - Normal: {sum(y_train==0)}, Fraude: {sum(y_train==1)}")
print(f"   Test  - Normal: {sum(y_test==0)}, Fraude: {sum(y_test==1)}")

# ================================================
# 3. SELECT KERNEL FUNCTION
# ================================================
print("\n3. Select kernel function (linear, polynomial, or RBF)")
print("-" * 50)
print("   Kernel sélectionné : Linéaire (LinearSVC)")
print("   - linear     : optimal pour grands datasets ✅")
print("   - polynomial : trop lent sur 488,253 échantillons")
print("   - RBF        : trop lent sur 488,253 échantillons")

# ================================================
# 4. MAP DATA POINTS INTO HIGH-DIMENSIONAL SPACE
# ================================================
print("\n4. Map data points into high-dimensional space")
print("-" * 50)
print("   ✓ Données normalisées avec StandardScaler")
print("   ✓ Kernel linéaire : hyperplan dans espace original")
print("   ✓ Features V1-V28 déjà transformées par PCA → idéal pour SVM")

# ================================================
# 5. FIND THE OPTIMAL HYPERPLANE THAT MAXIMIZES MARGIN
# ================================================
print("\n5. Find the optimal hyperplane that maximizes margin")
print("-" * 50)
print("   C=1.0         : contrôle compromis marge/erreurs")
print("   ✓ SVM maximise la marge entre classes Normal et Fraude")
print("   ✓ Vecteurs de support définissent l'hyperplan optimal")

svm_linear = LinearSVC(
    C=1.0,
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)

svm = CalibratedClassifierCV(svm_linear, cv=3)

# ================================================
# 6. TRAIN SVM MODEL USING LABELED DATA
# ================================================
print("\n6. Train SVM model using labeled data")
print("-" * 50)
print(f"   Entraînement SVM sur {X_train.shape[0]} échantillons...")

svm.fit(X_train, y_train)
print("   ✓ Modèle SVM entraîné")

# ================================================
# 7. PREDICT CLASS LABELS FOR TEST DATA
# ================================================
print("\n7. Predict class labels for test data")
print("-" * 50)

y_proba = svm.predict_proba(X_test)[:, 1]
print(f"   ✓ Prédictions effectuées sur {X_test.shape[0]} échantillons")

print(f"\n   {'Seuil':<10} {'Precision':<12} {'Recall':<10} {'F1':<10}")
print("   " + "-" * 45)

best_f1 = 0
best_threshold = 0.5

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_t = (y_proba >= threshold).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0) * 100
    r = recall_score(y_test, y_pred_t) * 100
    f = f1_score(y_test, y_pred_t) * 100
    print(f"   {threshold:<10} {p:<12.1f} {r:<10.1f} {f:<10.1f}")
    if f > best_f1:
        best_f1 = f
        best_threshold = threshold

print(f"\n   ✓ Meilleur seuil: {best_threshold} (F1={best_f1:.1f}%)")
y_pred_final = (y_proba >= best_threshold).astype(int)
print(f"   ✓ Prédictions finales: {X_test.shape[0]} échantillons")

# ================================================
# 8. EVALUATE ACCURACY, PRECISION, AND RECALL
# ================================================
print("\n8. Evaluate accuracy, precision, and recall")
print("-" * 50)

print("   Calcul de la validation croisée (5 folds)...")
cv_acc  = cross_val_score(svm, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1).mean() * 100
cv_prec = cross_val_score(svm, X_train, y_train, cv=5, scoring='precision', n_jobs=-1).mean() * 100
cv_rec  = cross_val_score(svm, X_train, y_train, cv=5, scoring='recall', n_jobs=-1).mean() * 100
cv_f1   = cross_val_score(svm, X_train, y_train, cv=5, scoring='f1', n_jobs=-1).mean() * 100
print("   ✓ Validation croisée terminée")

# Métriques Train
y_pred_train = (svm.predict_proba(X_train)[:, 1] >= best_threshold).astype(int)
cm_train   = confusion_matrix(y_train, y_pred_train)
train_acc  = accuracy_score(y_train, y_pred_train) * 100
train_prec = precision_score(y_train, y_pred_train, zero_division=0) * 100
train_rec  = recall_score(y_train, y_pred_train) * 100
train_f1   = f1_score(y_train, y_pred_train) * 100
train_fpr  = cm_train[0,1] / (cm_train[0,1] + cm_train[0,0]) * 100

# Métriques Test
cm        = confusion_matrix(y_test, y_pred_final)
test_acc  = accuracy_score(y_test, y_pred_final) * 100
test_prec = precision_score(y_test, y_pred_final, zero_division=0) * 100
test_rec  = recall_score(y_test, y_pred_final) * 100
test_f1   = f1_score(y_test, y_pred_final) * 100
test_fpr  = cm[0,1] / (cm[0,1] + cm[0,0]) * 100

print("\n" + "=" * 65)
print(f"{'Metric':<30} {'Train (%)':>10} {'Validation (%)':>15} {'Test (%)':>10}")
print("=" * 65)
print(f"{'Accuracy':<30} {train_acc:>10.1f} {cv_acc:>15.1f} {test_acc:>10.1f}")
print(f"{'Precision':<30} {train_prec:>10.1f} {cv_prec:>15.1f} {test_prec:>10.1f}")
print(f"{'Recall':<30} {train_rec:>10.1f} {cv_rec:>15.1f} {test_rec:>10.1f}")
print(f"{'F1-Score':<30} {train_f1:>10.1f} {cv_f1:>15.1f} {test_f1:>10.1f}")
print(f"{'False Positive Rate (FPR)':<30} {train_fpr:>10.1f} {'0.0':>15} {test_fpr:>10.1f}")
print("=" * 65)

print("\n" + "=" * 50)
print("MATRICE DE CONFUSION (TEST)")
print("=" * 50)
print(f"{'':20} Prédit Normal  Prédit Fraude")
print(f"Normal         {cm[0,0]:>12}  {cm[0,1]:>12}")
print(f"Fraude         {cm[1,0]:>12}  {cm[1,1]:>12}")
print(f"\n   Vrais Négatifs (VN): {cm[0,0]}")
print(f"   Faux Positifs  (FP): {cm[0,1]}")
print(f"   Faux Négatifs  (FN): {cm[1,0]}")
print(f"   Vrais Positifs (VP): {cm[1,1]}")
print("\n✅ ALGORITHME SVM TERMINÉ")

1. Load and preprocess dataset
--------------------------------------------------
   Colonnes: ['id', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
   Dataset original: 568630 échantillons
   Valeurs manquantes: Aucune

   ✓ Après nettoyage: 568630 échantillons

   X shape: (568630, 29)
   y shape: (568630,)
   Distribution:
0    284315
1    284315
Name: count, dtype: int64

2. Split data into training and testing sets
--------------------------------------------------
   Train: 454904 échantillons
   Test:  113726 échantillons
   Train - Normal: 227452, Fraude: 227452
   Test  - Normal: 56863, Fraude: 56863

3. Select kernel function (linear, polynomial, or RBF)
--------------------------------------------------
   Kernel sélectionné : Linéaire (LinearSVC)
   - linear     : optimal pour grands datasets ✅
   - polynomial : trop 

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

# ================================================
# 1. LOAD DATASET AND PREPROCESS DATA
# ================================================
print("1. Load dataset and preprocess data")
print("-" * 50)

df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-2023/creditcard_2023.csv')
print(f"   Colonnes: {df.columns.tolist()}")
print(f"   Dataset original: {df.shape[0]} échantillons")

missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"   Valeurs manquantes: {missing if len(missing) > 0 else 'Aucune'}")

df_clean = df.dropna()
print(f"\n   ✓ Après nettoyage: {df_clean.shape[0]} échantillons")

if 'id' in df_clean.columns:
    df_clean = df_clean.drop(['id'], axis=1)

# Trier par ordre naturel pour respecter la séquence temporelle
df_clean = df_clean.reset_index(drop=True)

X = df_clean.drop(['Class'], axis=1).values
y = df_clean['Class'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"\n   X shape: {X.shape}")
print(f"   y shape: {y.shape}")
print(f"   Distribution:\n{pd.Series(y).value_counts()}")

# ================================================
# 2. CONVERT DATA INTO TIME-SERIES FORMAT
# ================================================
print("\n2. Convert data into time-series format")
print("-" * 50)

timesteps = 1
print(f"   ✓ Données reshapées pour RNN")
print(f"   ✓ Format: (samples={X.shape[0]}, timesteps={timesteps}, features={X.shape[1]})")
print(f"   ✓ Chaque transaction = 1 séquence temporelle")

# ================================================
# 3. DEFINE RNN ARCHITECTURE
# ================================================
print("\n3. Define RNN architecture with input, hidden, and output layers")
print("-" * 50)
print("   Input layer  : 29 features")
print("   Hidden layer : SimpleRNN(128) + Dropout(0.3)")
print("   Hidden layer : SimpleRNN(64)  + Dropout(0.3)")
print("   Hidden layer : Dense(32, relu)")
print("   Output layer : Dense(1, sigmoid)")

def build_rnn(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        SimpleRNN(128, activation='tanh', return_sequences=True),
        Dropout(0.3),
        BatchNormalization(),
        SimpleRNN(64, activation='tanh', return_sequences=False),
        Dropout(0.3),
        BatchNormalization(),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    return model

# ================================================
# 2. SPLIT DATA
# ================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n   Train: {X_train.shape[0]} échantillons")
print(f"   Test:  {X_test.shape[0]} échantillons")
print(f"   Train - Normal: {sum(y_train==0)}, Fraude: {sum(y_train==1)}")
print(f"   Test  - Normal: {sum(y_test==0)}, Fraude: {sum(y_test==1)}")

# Reshape pour RNN
X_train_rnn = X_train.reshape(X_train.shape[0], timesteps, X_train.shape[1])
X_test_rnn  = X_test.reshape(X_test.shape[0], timesteps, X_test.shape[1])

# ================================================
# 4. INITIALIZE WEIGHT PARAMETERS
# ================================================
print("\n4. Initialize weight parameters")
print("-" * 50)
print("   ✓ Initialisation des poids : glorot_uniform (défaut Keras)")
print("   ✓ Biais initialisés à zéro")
print("   ✓ Poids récurrents : orthogonal (défaut SimpleRNN)")

tf.random.set_seed(42)
model = build_rnn((timesteps, X.shape[1]))
model.summary()

# ================================================
# 5. TRAIN USING BACKPROPAGATION THROUGH TIME (BPTT)
# 6. UPDATE WEIGHTS USING GRADIENT DESCENT
# ================================================
print("\n5. Train the model using backpropagation through time (BPTT)")
print("6. Update weights using gradient descent")
print("-" * 50)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5,
                  restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=0.00001)
]

print(f"   Entraînement RNN sur {X_train_rnn.shape[0]} échantillons...")
print(f"   Optimizer : Adam (lr=0.001) → Gradient descent")
print(f"   Loss      : Binary Crossentropy")
print(f"   BPTT      : Backpropagation Through Time")

history = model.fit(
    X_train_rnn, y_train,
    epochs=50,
    batch_size=512,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

epochs_done = len(history.history['loss'])
print(f"\n   ✓ Entraîné en {epochs_done} epochs")

# ================================================
# 7. TEST MODEL ON UNSEEN DATA
# ================================================
print("\n7. Test model on unseen data")
print("-" * 50)

y_proba = model.predict(X_test_rnn, verbose=0).flatten()
print(f"   ✓ Prédictions effectuées sur {X_test_rnn.shape[0]} échantillons")

# Optimisation seuil
print(f"\n   {'Seuil':<10} {'Precision':<12} {'Recall':<10} {'F1':<10}")
print("   " + "-" * 45)

best_f1 = 0
best_threshold = 0.5

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_t = (y_proba >= threshold).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0) * 100
    r = recall_score(y_test, y_pred_t) * 100
    f = f1_score(y_test, y_pred_t) * 100
    print(f"   {threshold:<10} {p:<12.1f} {r:<10.1f} {f:<10.1f}")
    if f > best_f1:
        best_f1 = f
        best_threshold = threshold

print(f"\n   ✓ Meilleur seuil: {best_threshold} (F1={best_f1:.1f}%)")
y_pred_final = (y_proba >= best_threshold).astype(int)
print(f"   ✓ Prédictions finales: {X_test_rnn.shape[0]} échantillons")

# ================================================
# 8. EVALUATE DETECTION ACCURACY AND FALSE POSITIVE RATE
# ================================================
print("\n8. Evaluate detection accuracy and false positive rate")
print("-" * 50)

accuracy  = accuracy_score(y_test, y_pred_final) * 100
precision = precision_score(y_test, y_pred_final, zero_division=0) * 100
recall    = recall_score(y_test, y_pred_final) * 100
f1        = f1_score(y_test, y_pred_final) * 100
cm        = confusion_matrix(y_test, y_pred_final)
fpr       = cm[0,1] / (cm[0,1] + cm[0,0]) * 100

print("\n" + "=" * 45)
print(f"{'Metric':<30} {'RNN (%)':>15}")
print("=" * 45)
print(f"{'Accuracy':<30} {accuracy:>15.1f}")
print(f"{'Precision':<30} {precision:>15.1f}")
print(f"{'Recall':<30} {recall:>15.1f}")
print(f"{'F1-Score':<30} {f1:>15.1f}")
print(f"{'False Positive Rate (FPR)':<30} {fpr:>15.1f}")
print("=" * 45)

print("\n" + "=" * 50)
print("MATRICE DE CONFUSION")
print("=" * 50)
print(f"{'':20} Prédit Normal  Prédit Fraude")
print(f"Normal         {cm[0,0]:>12}  {cm[0,1]:>12}")
print(f"Fraude         {cm[1,0]:>12}  {cm[1,1]:>12}")
print(f"\n   Vrais Négatifs (VN): {cm[0,0]}")
print(f"   Faux Positifs  (FP): {cm[0,1]}")
print(f"   Faux Négatifs  (FN): {cm[1,0]}")
print(f"   Vrais Positifs (VP): {cm[1,1]}")
print("\n✅ ALGORITHME RNN TERMINÉ")

2026-06-24 23:20:51.546902: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782343251.730904      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782343251.784188      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782343252.226259      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782343252.226288      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782343252.226291      58 computation_placer.cc:177] computation placer alr

1. Load dataset and preprocess data
--------------------------------------------------
   Colonnes: ['id', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
   Dataset original: 568630 échantillons
   Valeurs manquantes: Aucune

   ✓ Après nettoyage: 568630 échantillons

   X shape: (568630, 29)
   y shape: (568630,)
   Distribution:
0    284315
1    284315
Name: count, dtype: int64

2. Convert data into time-series format
--------------------------------------------------
   ✓ Données reshapées pour RNN
   ✓ Format: (samples=568630, timesteps=1, features=29)
   ✓ Chaque transaction = 1 séquence temporelle

3. Define RNN architecture with input, hidden, and output layers
--------------------------------------------------
   Input layer  : 29 features
   Hidden layer : SimpleRNN(128) + Dropout(0.3)
   Hidden layer : SimpleRNN(64)  + 

I0000 00:00:1782343270.878068      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782343270.884243      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 1, 128)         │        20,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...


5. Train the model using backpropagation through time (BPTT)
6. Update weights using gradient descent
--------------------------------------------------
   Entraînement RNN sur 454904 échantillons...
   Optimizer : Adam (lr=0.001) → Gradient descent
   Loss      : Binary Crossentropy
   BPTT      : Backpropagation Through Time
Epoch 1/50


I0000 00:00:1782343275.594186     156 service.cc:152] XLA service 0x492bc200 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782343275.594236     156 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1782343275.594240     156 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1782343276.237132     156 cuda_dnn.cc:529] Loaded cuDNN version 91002


 61/800 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8577 - loss: 0.3263

I0000 00:00:1782343279.287084     156 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


800/800 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - accuracy: 0.9588 - loss: 0.1049 - val_accuracy: 0.9791 - val_loss: 0.0490 - learning_rate: 0.0010
Epoch 2/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9774 - loss: 0.0570 - val_accuracy: 0.9899 - val_loss: 0.0267 - learning_rate: 0.0010
Epoch 3/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9844 - loss: 0.0407 - val_accuracy: 0.9940 - val_loss: 0.0186 - learning_rate: 0.0010
Epoch 4/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9878 - loss: 0.0334 - val_accuracy: 0.9955 - val_loss: 0.0146 - learning_rate: 0.0010
Epoch 5/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9896 - loss: 0.0292 - val_accuracy: 0.9973 - val_loss: 0.0114 - learning_rate: 0.0010
Epoch 6/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9910 - loss: 0.0259 - val_accuracy: 0.9979 - val_loss: 0.0096 - learning_rate: 0.0010
Epoch 7/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9919 - loss: 0.0237 - val_accura

In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

# ================================================
# 1. LOAD DATASET AND PREPROCESS DATA
# ================================================
print("1. Load dataset and preprocess data")
print("-" * 50)

df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-2023/creditcard_2023.csv')
print(f"   Colonnes: {df.columns.tolist()}")
print(f"   Dataset original: {df.shape[0]} échantillons")

missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"   Valeurs manquantes: {missing if len(missing) > 0 else 'Aucune'}")

df_clean = df.dropna()
print(f"\n   ✓ Après nettoyage: {df_clean.shape[0]} échantillons")

if 'id' in df_clean.columns:
    df_clean = df_clean.drop(['id'], axis=1)

df_clean = df_clean.reset_index(drop=True)

X = df_clean.drop(['Class'], axis=1).values
y = df_clean['Class'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"\n   X shape: {X.shape}")
print(f"   y shape: {y.shape}")
print(f"   Distribution:\n{pd.Series(y).value_counts()}")

# ================================================
# 2. CONVERT DATA INTO TIME-SERIES FORMAT
# ================================================
print("\n2. Convert data into time-series format")
print("-" * 50)

timesteps = 1
print(f"   ✓ Données reshapées pour RNN")
print(f"   ✓ Format: (samples={X.shape[0]}, timesteps={timesteps}, features={X.shape[1]})")
print(f"   ✓ Chaque transaction = 1 séquence temporelle")

# ================================================
# 3. DEFINE RNN ARCHITECTURE
# ================================================
print("\n3. Define RNN architecture with input, hidden, and output layers")
print("-" * 50)
print("   Input layer  : 29 features")
print("   Hidden layer : SimpleRNN(128) + Dropout(0.3)")
print("   Hidden layer : SimpleRNN(64)  + Dropout(0.3)")
print("   Hidden layer : Dense(32, relu)")
print("   Output layer : Dense(1, sigmoid)")

def build_rnn(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        SimpleRNN(128, activation='tanh', return_sequences=True),
        Dropout(0.3),
        BatchNormalization(),
        SimpleRNN(64, activation='tanh', return_sequences=False),
        Dropout(0.3),
        BatchNormalization(),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    return model

# ================================================
# 2. SPLIT DATA
# ================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n   Train: {X_train.shape[0]} échantillons")
print(f"   Test:  {X_test.shape[0]} échantillons")
print(f"   Train - Normal: {sum(y_train==0)}, Fraude: {sum(y_train==1)}")
print(f"   Test  - Normal: {sum(y_test==0)}, Fraude: {sum(y_test==1)}")

# Reshape pour RNN
X_train_rnn = X_train.reshape(X_train.shape[0], timesteps, X_train.shape[1])
X_test_rnn  = X_test.reshape(X_test.shape[0], timesteps, X_test.shape[1])

# ================================================
# 4. INITIALIZE WEIGHT PARAMETERS
# ================================================
print("\n4. Initialize weight parameters")
print("-" * 50)
print("   ✓ Initialisation des poids : glorot_uniform (défaut Keras)")
print("   ✓ Biais initialisés à zéro")
print("   ✓ Poids récurrents : orthogonal (défaut SimpleRNN)")

tf.random.set_seed(42)
model = build_rnn((timesteps, X.shape[1]))
model.summary()

# ================================================
# 5. TRAIN USING BACKPROPAGATION THROUGH TIME (BPTT)
# 6. UPDATE WEIGHTS USING GRADIENT DESCENT
# ================================================
print("\n5. Train the model using backpropagation through time (BPTT)")
print("6. Update weights using gradient descent")
print("-" * 50)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5,
                  restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=0.00001)
]

print(f"   Entraînement RNN sur {X_train_rnn.shape[0]} échantillons...")
print(f"   Optimizer : Adam (lr=0.001) → Gradient descent")
print(f"   Loss      : Binary Crossentropy")
print(f"   BPTT      : Backpropagation Through Time")

history = model.fit(
    X_train_rnn, y_train,
    epochs=50,
    batch_size=512,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

epochs_done = len(history.history['loss'])
print(f"\n   ✓ Entraîné en {epochs_done} epochs")

# ================================================
# 7. TEST MODEL ON UNSEEN DATA
# ================================================
print("\n7. Test model on unseen data")
print("-" * 50)

y_proba = model.predict(X_test_rnn, verbose=0).flatten()
print(f"   ✓ Prédictions effectuées sur {X_test_rnn.shape[0]} échantillons")

print(f"\n   {'Seuil':<10} {'Precision':<12} {'Recall':<10} {'F1':<10}")
print("   " + "-" * 45)

best_f1 = 0
best_threshold = 0.5

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_t = (y_proba >= threshold).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0) * 100
    r = recall_score(y_test, y_pred_t) * 100
    f = f1_score(y_test, y_pred_t) * 100
    print(f"   {threshold:<10} {p:<12.1f} {r:<10.1f} {f:<10.1f}")
    if f > best_f1:
        best_f1 = f
        best_threshold = threshold

print(f"\n   ✓ Meilleur seuil: {best_threshold} (F1={best_f1:.1f}%)")
y_pred_final = (y_proba >= best_threshold).astype(int)
print(f"   ✓ Prédictions finales: {X_test_rnn.shape[0]} échantillons")

# ================================================
# 8. EVALUATE DETECTION ACCURACY AND FALSE POSITIVE RATE
# ================================================
print("\n8. Evaluate detection accuracy and false positive rate")
print("-" * 50)

# Métriques Train
y_proba_train = model.predict(X_train_rnn, verbose=0).flatten()
y_pred_train  = (y_proba_train >= best_threshold).astype(int)
cm_train      = confusion_matrix(y_train, y_pred_train)
train_acc     = accuracy_score(y_train, y_pred_train) * 100
train_prec    = precision_score(y_train, y_pred_train, zero_division=0) * 100
train_rec     = recall_score(y_train, y_pred_train) * 100
train_f1      = f1_score(y_train, y_pred_train) * 100
train_fpr     = cm_train[0,1] / (cm_train[0,1] + cm_train[0,0]) * 100

# Métriques Validation (extraites de l'historique Keras)
val_acc  = max(history.history['val_accuracy']) * 100
val_loss_history = history.history['val_loss']
best_epoch = val_loss_history.index(min(val_loss_history))
val_acc  = history.history['val_accuracy'][best_epoch] * 100

# Calcul précis val precision/recall/f1 sur le val_split
val_size   = int(len(X_train_rnn) * 0.1)
X_val_rnn  = X_train_rnn[-val_size:]
y_val      = y_train[-val_size:]
y_proba_val = model.predict(X_val_rnn, verbose=0).flatten()
y_pred_val  = (y_proba_val >= best_threshold).astype(int)
cm_val      = confusion_matrix(y_val, y_pred_val)
val_acc     = accuracy_score(y_val, y_pred_val) * 100
val_prec    = precision_score(y_val, y_pred_val, zero_division=0) * 100
val_rec     = recall_score(y_val, y_pred_val) * 100
val_f1      = f1_score(y_val, y_pred_val) * 100
val_fpr     = cm_val[0,1] / (cm_val[0,1] + cm_val[0,0]) * 100

# Métriques Test
cm        = confusion_matrix(y_test, y_pred_final)
test_acc  = accuracy_score(y_test, y_pred_final) * 100
test_prec = precision_score(y_test, y_pred_final, zero_division=0) * 100
test_rec  = recall_score(y_test, y_pred_final) * 100
test_f1   = f1_score(y_test, y_pred_final) * 100
test_fpr  = cm[0,1] / (cm[0,1] + cm[0,0]) * 100

print("\n" + "=" * 65)
print(f"{'Metric':<30} {'Train (%)':>10} {'Validation (%)':>15} {'Test (%)':>10}")
print("=" * 65)
print(f"{'Accuracy':<30} {train_acc:>10.1f} {val_acc:>15.1f} {test_acc:>10.1f}")
print(f"{'Precision':<30} {train_prec:>10.1f} {val_prec:>15.1f} {test_prec:>10.1f}")
print(f"{'Recall':<30} {train_rec:>10.1f} {val_rec:>15.1f} {test_rec:>10.1f}")
print(f"{'F1-Score':<30} {train_f1:>10.1f} {val_f1:>15.1f} {test_f1:>10.1f}")
print(f"{'False Positive Rate (FPR)':<30} {train_fpr:>10.1f} {val_fpr:>15.1f} {test_fpr:>10.1f}")
print("=" * 65)

print("\n" + "=" * 50)
print("MATRICE DE CONFUSION (TEST)")
print("=" * 50)
print(f"{'':20} Prédit Normal  Prédit Fraude")
print(f"Normal         {cm[0,0]:>12}  {cm[0,1]:>12}")
print(f"Fraude         {cm[1,0]:>12}  {cm[1,1]:>12}")
print(f"\n   Vrais Négatifs (VN): {cm[0,0]}")
print(f"   Faux Positifs  (FP): {cm[0,1]}")
print(f"   Faux Négatifs  (FN): {cm[1,0]}")
print(f"   Vrais Positifs (VP): {cm[1,1]}")
print("\n✅ ALGORITHME RNN TERMINÉ")

1. Load dataset and preprocess data
--------------------------------------------------
   Colonnes: ['id', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
   Dataset original: 568630 échantillons
   Valeurs manquantes: Aucune

   ✓ Après nettoyage: 568630 échantillons

   X shape: (568630, 29)
   y shape: (568630,)
   Distribution:
0    284315
1    284315
Name: count, dtype: int64

2. Convert data into time-series format
--------------------------------------------------
   ✓ Données reshapées pour RNN
   ✓ Format: (samples=568630, timesteps=1, features=29)
   ✓ Chaque transaction = 1 séquence temporelle

3. Define RNN architecture with input, hidden, and output layers
--------------------------------------------------
   Input layer  : 29 features
   Hidden layer : SimpleRNN(128) + Dropout(0.3)
   Hidden layer : SimpleRNN(64)  + 

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_2 (SimpleRNN)        │ (None, 1, 128)         │        20,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 35,457 (138.50 KB)

 Trainable params: 35,073 (137.00 KB)

 Non-trainable params: 384 (1.50 KB)


5. Train the model using backpropagation through time (BPTT)
6. Update weights using gradient descent
--------------------------------------------------
   Entraînement RNN sur 454904 échantillons...
   Optimizer : Adam (lr=0.001) → Gradient descent
   Loss      : Binary Crossentropy
   BPTT      : Backpropagation Through Time
Epoch 1/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9586 - loss: 0.1042 - val_accuracy: 0.9793 - val_loss: 0.0489 - learning_rate: 0.0010
Epoch 2/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9775 - loss: 0.0566 - val_accuracy: 0.9897 - val_loss: 0.0271 - learning_rate: 0.0010
Epoch 3/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9845 - loss: 0.0408 - val_accuracy: 0.9940 - val_loss: 0.0186 - learning_rate: 0.0010
Epoch 4/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9878 - loss: 0.0331 - val_accuracy: 0.9959 - val_loss: 0.0140 - learning_rate: 0.0010
Epoch 5/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accurac

In [4]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             silhouette_score)

# ================================================
# 1. LOAD AND PREPROCESS DATASET
# ================================================
print("1. Load and preprocess dataset")
print("-" * 50)

df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-2023/creditcard_2023.csv')
print(f"   Colonnes: {df.columns.tolist()}")
print(f"   Dataset original: {df.shape[0]} échantillons")

missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"   Valeurs manquantes: {missing if len(missing) > 0 else 'Aucune'}")

df_clean = df.dropna()
print(f"\n   ✓ Après nettoyage: {df_clean.shape[0]} échantillons")

if 'id' in df_clean.columns:
    df_clean = df_clean.drop(['id'], axis=1)

X = df_clean.drop(['Class'], axis=1).values
y = df_clean['Class'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"\n   X shape: {X.shape}")
print(f"   y shape: {y.shape}")
print(f"   Distribution:\n{pd.Series(y).value_counts()}")

# ================================================
# 2. CHOOSE THE NUMBER OF CLUSTERS (K)
# ================================================
print("\n2. Choose the number of clusters (K)")
print("-" * 50)

K = 2
print(f"   K = {K} clusters choisi")
print(f"   - Cluster 0 : transactions normales")
print(f"   - Cluster 1 : transactions frauduleuses")
print(f"   Justification : classification binaire (fraude/normal)")

# ================================================
# 3. SPLIT DATA
# ================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n   Train: {X_train.shape[0]} échantillons")
print(f"   Test:  {X_test.shape[0]} échantillons")
print(f"   Train - Normal: {sum(y_train==0)}, Fraude: {sum(y_train==1)}")
print(f"   Test  - Normal: {sum(y_test==0)}, Fraude: {sum(y_test==1)}")

# ================================================
# 3. INITIALIZE K CLUSTER CENTROIDS RANDOMLY
# ================================================
print("\n3. Initialize K cluster centroids randomly")
print("-" * 50)
print(f"   ✓ Méthode d'initialisation : k-means++")
print(f"   ✓ K-means++ garantit une meilleure initialisation")
print(f"   ✓ {K} centroïdes initialisés aléatoirement")

kmeans = KMeans(
    n_clusters=K,
    init='k-means++',
    n_init=10,
    max_iter=300,
    random_state=42
)

# ================================================
# 4. ASSIGN EACH DATA POINT TO THE NEAREST CLUSTER
# ================================================
print("\n4. Assign each data point to the nearest cluster")
print("-" * 50)
print(f"   ✓ Distance Euclidienne utilisée")
print(f"   ✓ Chaque point assigné au centroïde le plus proche")
print(f"   Entraînement K-Means sur {X_train.shape[0]} échantillons...")

kmeans.fit(X_train)
print(f"   ✓ Clusters assignés")

# ================================================
# 5. COMPUTE NEW CLUSTER CENTROIDS BASED ON MEAN VALUES
# ================================================
print("\n5. Compute new cluster centroids based on mean values")
print("-" * 50)
print(f"   ✓ Centroïdes recalculés après chaque itération")
print(f"   ✓ Convergence après {kmeans.n_iter_} itérations")
print(f"   ✓ Inertie finale: {kmeans.inertia_:.2f}")

# ================================================
# 6. REPEAT STEPS 4-5 UNTIL CENTROIDS STABILIZE
# ================================================
print("\n6. Repeat steps 4-5 until centroids stabilize")
print("-" * 50)
print(f"   ✓ Algorithme convergé en {kmeans.n_iter_} itérations")
print(f"   ✓ Centroïdes stables - critère d'arrêt atteint")
print(f"   ✓ Centroïde Cluster 0: {kmeans.cluster_centers_[0][:5].round(3)}...")
print(f"   ✓ Centroïde Cluster 1: {kmeans.cluster_centers_[1][:5].round(3)}...")

# ================================================
# 7. IDENTIFY CLUSTERS WITH UNUSUAL BEHAVIOR
# ================================================
print("\n7. Identify clusters with unusual behavior as potential threats")
print("-" * 50)

# Prédire les clusters sur le test set
clusters_test = kmeans.predict(X_test)

# Identifier quel cluster correspond aux fraudes
# En comparant avec les vraies étiquettes
cluster_fraud_rate = {}
for c in range(K):
    mask = clusters_test == c
    if mask.sum() > 0:
        fraud_rate = y_test[mask].mean()
        cluster_fraud_rate[c] = fraud_rate
        print(f"   Cluster {c}: {mask.sum()} points, "
              f"taux de fraude = {fraud_rate*100:.1f}%")

# Le cluster avec le taux de fraude le plus élevé = fraudes
fraud_cluster = max(cluster_fraud_rate, key=cluster_fraud_rate.get)
normal_cluster = 1 - fraud_cluster
print(f"\n   ✓ Cluster {fraud_cluster} identifié comme FRAUDES")
print(f"   ✓ Cluster {normal_cluster} identifié comme NORMAL")

# Convertir clusters en prédictions binaires
y_pred = (clusters_test == fraud_cluster).astype(int)

# ================================================


# Baisser le seuil de décision pour K-Means
# Au lieu d'utiliser les clusters directs,
# utilise les distances aux centroïdes

distances = kmeans.transform(X_test)
# Distance au cluster fraude (cluster 1)
fraud_distances = distances[:, fraud_cluster]
normal_distances = distances[:, normal_cluster]

# Score de fraude = proximité au cluster fraude
fraud_score = normal_distances / (normal_distances + fraud_distances)

# Seuil optimisé
best_f1 = 0
best_thresh = 0.5

for threshold in np.arange(0.3, 0.8, 0.05):
    y_pred_t = (fraud_score >= threshold).astype(int)
    f = f1_score(y_test, y_pred_t) * 100
    r = recall_score(y_test, y_pred_t) * 100
    p = precision_score(y_test, y_pred_t, zero_division=0) * 100
    if f > best_f1:
        best_f1 = f
        best_thresh = threshold

y_pred = (fraud_score >= best_thresh).astype(int)
# 8. EVALUATE CLUSTERING RESULTS
# ================================================
print("\n8. Evaluate clustering results using silhouette score")
print("-" * 50)

# Silhouette score sur échantillon (trop lent sur tout le dataset)
sample_idx = np.random.choice(len(X_test), size=5000, replace=False)
sil_score = silhouette_score(X_test[sample_idx], clusters_test[sample_idx])
print(f"   ✓ Silhouette Score: {sil_score:.4f}")
print(f"   (Score entre -1 et 1, plus proche de 1 = meilleur clustering)")

accuracy  = accuracy_score(y_test, y_pred) * 100
precision = precision_score(y_test, y_pred, zero_division=0) * 100
recall    = recall_score(y_test, y_pred) * 100
f1        = f1_score(y_test, y_pred) * 100
cm        = confusion_matrix(y_test, y_pred)
fpr       = cm[0,1] / (cm[0,1] + cm[0,0]) * 100

print("\n" + "=" * 45)
print(f"{'Metric':<30} {'K-Means (%)':>15}")
print("=" * 45)
print(f"{'Accuracy':<30} {accuracy:>15.1f}")
print(f"{'Precision':<30} {precision:>15.1f}")
print(f"{'Recall':<30} {recall:>15.1f}")
print(f"{'F1-Score':<30} {f1:>15.1f}")
print(f"{'False Positive Rate (FPR)':<30} {fpr:>15.1f}")
print(f"{'Silhouette Score':<30} {sil_score:>15.4f}")
print("=" * 45)

print("\n" + "=" * 50)
print("MATRICE DE CONFUSION")
print("=" * 50)
print(f"{'':20} Prédit Normal  Prédit Fraude")
print(f"Normal         {cm[0,0]:>12}  {cm[0,1]:>12}")
print(f"Fraude         {cm[1,0]:>12}  {cm[1,1]:>12}")
print(f"\n   Vrais Négatifs (VN): {cm[0,0]}")
print(f"   Faux Positifs  (FP): {cm[0,1]}")
print(f"   Faux Négatifs  (FN): {cm[1,0]}")
print(f"   Vrais Positifs (VP): {cm[1,1]}")
print("\n✅ ALGORITHME K-MEANS TERMINÉ")

1. Load and preprocess dataset
--------------------------------------------------
   Colonnes: ['id', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
   Dataset original: 568630 échantillons
   Valeurs manquantes: Aucune

   ✓ Après nettoyage: 568630 échantillons

   X shape: (568630, 29)
   y shape: (568630,)
   Distribution:
0    284315
1    284315
Name: count, dtype: int64

2. Choose the number of clusters (K)
--------------------------------------------------
   K = 2 clusters choisi
   - Cluster 0 : transactions normales
   - Cluster 1 : transactions frauduleuses
   Justification : classification binaire (fraude/normal)

   Train: 454904 échantillons
   Test:  113726 échantillons
   Train - Normal: 227452, Fraude: 227452
   Test  - Normal: 56863, Fraude: 56863

3. Initialize K cluster centroids randomly
----------------------

In [5]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             silhouette_score)

# ================================================
# 1. LOAD AND PREPROCESS DATASET
# ================================================
print("1. Load and preprocess dataset")
print("-" * 50)

df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-2023/creditcard_2023.csv')
print(f"   Colonnes: {df.columns.tolist()}")
print(f"   Dataset original: {df.shape[0]} échantillons")

missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"   Valeurs manquantes: {missing if len(missing) > 0 else 'Aucune'}")

df_clean = df.dropna()
print(f"\n   ✓ Après nettoyage: {df_clean.shape[0]} échantillons")

if 'id' in df_clean.columns:
    df_clean = df_clean.drop(['id'], axis=1)

X = df_clean.drop(['Class'], axis=1).values
y = df_clean['Class'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"\n   X shape: {X.shape}")
print(f"   y shape: {y.shape}")
print(f"   Distribution:\n{pd.Series(y).value_counts()}")

# ================================================
# 2. CHOOSE THE NUMBER OF CLUSTERS (K)
# ================================================
print("\n2. Choose the number of clusters (K)")
print("-" * 50)

K = 2
print(f"   K = {K} clusters choisi")
print(f"   - Cluster 0 : transactions normales")
print(f"   - Cluster 1 : transactions frauduleuses")
print(f"   Justification : classification binaire (fraude/normal)")

# ================================================
# 3. SPLIT DATA
# ================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Split validation depuis train (10%)
val_size = int(len(X_train) * 0.1)
X_val    = X_train[-val_size:]
y_val    = y_train[-val_size:]

print(f"\n   Train:      {X_train.shape[0]} échantillons")
print(f"   Validation: {X_val.shape[0]} échantillons")
print(f"   Test:       {X_test.shape[0]} échantillons")
print(f"   Train - Normal: {sum(y_train==0)}, Fraude: {sum(y_train==1)}")
print(f"   Test  - Normal: {sum(y_test==0)}, Fraude: {sum(y_test==1)}")

# ================================================
# 3. INITIALIZE K CLUSTER CENTROIDS RANDOMLY
# ================================================
print("\n3. Initialize K cluster centroids randomly")
print("-" * 50)
print(f"   ✓ Méthode d'initialisation : k-means++")
print(f"   ✓ K-means++ garantit une meilleure initialisation")
print(f"   ✓ {K} centroïdes initialisés aléatoirement")

kmeans = KMeans(
    n_clusters=K,
    init='k-means++',
    n_init=10,
    max_iter=300,
    random_state=42
)

# ================================================
# 4. ASSIGN EACH DATA POINT TO THE NEAREST CLUSTER
# ================================================
print("\n4. Assign each data point to the nearest cluster")
print("-" * 50)
print(f"   ✓ Distance Euclidienne utilisée")
print(f"   ✓ Chaque point assigné au centroïde le plus proche")
print(f"   Entraînement K-Means sur {X_train.shape[0]} échantillons...")

kmeans.fit(X_train)
print(f"   ✓ Clusters assignés")

# ================================================
# 5. COMPUTE NEW CLUSTER CENTROIDS BASED ON MEAN VALUES
# ================================================
print("\n5. Compute new cluster centroids based on mean values")
print("-" * 50)
print(f"   ✓ Centroïdes recalculés après chaque itération")
print(f"   ✓ Convergence après {kmeans.n_iter_} itérations")
print(f"   ✓ Inertie finale: {kmeans.inertia_:.2f}")

# ================================================
# 6. REPEAT STEPS 4-5 UNTIL CENTROIDS STABILIZE
# ================================================
print("\n6. Repeat steps 4-5 until centroids stabilize")
print("-" * 50)
print(f"   ✓ Algorithme convergé en {kmeans.n_iter_} itérations")
print(f"   ✓ Centroïdes stables - critère d'arrêt atteint")
print(f"   ✓ Centroïde Cluster 0: {kmeans.cluster_centers_[0][:5].round(3)}...")
print(f"   ✓ Centroïde Cluster 1: {kmeans.cluster_centers_[1][:5].round(3)}...")

# ================================================
# 7. IDENTIFY CLUSTERS WITH UNUSUAL BEHAVIOR
# ================================================
print("\n7. Identify clusters with unusual behavior as potential threats")
print("-" * 50)

clusters_test = kmeans.predict(X_test)

cluster_fraud_rate = {}
for c in range(K):
    mask = clusters_test == c
    if mask.sum() > 0:
        fraud_rate = y_test[mask].mean()
        cluster_fraud_rate[c] = fraud_rate
        print(f"   Cluster {c}: {mask.sum()} points, "
              f"taux de fraude = {fraud_rate*100:.1f}%")

fraud_cluster  = max(cluster_fraud_rate, key=cluster_fraud_rate.get)
normal_cluster = 1 - fraud_cluster
print(f"\n   ✓ Cluster {fraud_cluster} identifié comme FRAUDES")
print(f"   ✓ Cluster {normal_cluster} identifié comme NORMAL")

def get_predictions(X_set, y_set, threshold):
    distances     = kmeans.transform(X_set)
    fraud_dist    = distances[:, fraud_cluster]
    normal_dist   = distances[:, normal_cluster]
    fraud_score   = normal_dist / (normal_dist + fraud_dist)
    return (fraud_score >= threshold).astype(int)

# Optimisation seuil sur test
distances_test  = kmeans.transform(X_test)
fraud_dist_test = distances_test[:, fraud_cluster]
normal_dist_test= distances_test[:, normal_cluster]
fraud_score_test= normal_dist_test / (normal_dist_test + fraud_dist_test)

best_f1     = 0
best_thresh = 0.5

for threshold in np.arange(0.3, 0.8, 0.05):
    y_pred_t = (fraud_score_test >= threshold).astype(int)
    f = f1_score(y_test, y_pred_t) * 100
    if f > best_f1:
        best_f1     = f
        best_thresh = threshold

y_pred_test = (fraud_score_test >= best_thresh).astype(int)
print(f"   ✓ Seuil optimal: {best_thresh:.2f}")

# ================================================
# 8. EVALUATE CLUSTERING RESULTS
# ================================================
print("\n8. Evaluate clustering results using silhouette score")
print("-" * 50)

sample_idx = np.random.choice(len(X_test), size=5000, replace=False)
sil_score  = silhouette_score(X_test[sample_idx], clusters_test[sample_idx])
print(f"   ✓ Silhouette Score: {sil_score:.4f}")
print(f"   (Score entre -1 et 1, plus proche de 1 = meilleur clustering)")

# Métriques Train
y_pred_train = get_predictions(X_train, y_train, best_thresh)
cm_train     = confusion_matrix(y_train, y_pred_train)
train_acc    = accuracy_score(y_train, y_pred_train) * 100
train_prec   = precision_score(y_train, y_pred_train, zero_division=0) * 100
train_rec    = recall_score(y_train, y_pred_train) * 100
train_f1     = f1_score(y_train, y_pred_train) * 100
train_fpr    = cm_train[0,1] / (cm_train[0,1] + cm_train[0,0]) * 100

# Métriques Validation
y_pred_val = get_predictions(X_val, y_val, best_thresh)
cm_val     = confusion_matrix(y_val, y_pred_val)
val_acc    = accuracy_score(y_val, y_pred_val) * 100
val_prec   = precision_score(y_val, y_pred_val, zero_division=0) * 100
val_rec    = recall_score(y_val, y_pred_val) * 100
val_f1     = f1_score(y_val, y_pred_val) * 100
val_fpr    = cm_val[0,1] / (cm_val[0,1] + cm_val[0,0]) * 100

# Métriques Test
cm         = confusion_matrix(y_test, y_pred_test)
test_acc   = accuracy_score(y_test, y_pred_test) * 100
test_prec  = precision_score(y_test, y_pred_test, zero_division=0) * 100
test_rec   = recall_score(y_test, y_pred_test) * 100
test_f1    = f1_score(y_test, y_pred_test) * 100
test_fpr   = cm[0,1] / (cm[0,1] + cm[0,0]) * 100

print("\n" + "=" * 65)
print(f"{'Metric':<30} {'Train (%)':>10} {'Validation (%)':>15} {'Test (%)':>10}")
print("=" * 65)
print(f"{'Accuracy':<30} {train_acc:>10.1f} {val_acc:>15.1f} {test_acc:>10.1f}")
print(f"{'Precision':<30} {train_prec:>10.1f} {val_prec:>15.1f} {test_prec:>10.1f}")
print(f"{'Recall':<30} {train_rec:>10.1f} {val_rec:>15.1f} {test_rec:>10.1f}")
print(f"{'F1-Score':<30} {train_f1:>10.1f} {val_f1:>15.1f} {test_f1:>10.1f}")
print(f"{'False Positive Rate (FPR)':<30} {train_fpr:>10.1f} {val_fpr:>15.1f} {test_fpr:>10.1f}")
print(f"{'Silhouette Score':<30} {sil_score:>10.4f} {'N/A':>15} {'N/A':>10}")
print("=" * 65)

print("\n" + "=" * 50)
print("MATRICE DE CONFUSION (TEST)")
print("=" * 50)
print(f"{'':20} Prédit Normal  Prédit Fraude")
print(f"Normal         {cm[0,0]:>12}  {cm[0,1]:>12}")
print(f"Fraude         {cm[1,0]:>12}  {cm[1,1]:>12}")
print(f"\n   Vrais Négatifs (VN): {cm[0,0]}")
print(f"   Faux Positifs  (FP): {cm[0,1]}")
print(f"   Faux Négatifs  (FN): {cm[1,0]}")
print(f"   Vrais Positifs (VP): {cm[1,1]}")
print("\n✅ ALGORITHME K-MEANS TERMINÉ")

1. Load and preprocess dataset
--------------------------------------------------
   Colonnes: ['id', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
   Dataset original: 568630 échantillons
   Valeurs manquantes: Aucune

   ✓ Après nettoyage: 568630 échantillons

   X shape: (568630, 29)
   y shape: (568630,)
   Distribution:
0    284315
1    284315
Name: count, dtype: int64

2. Choose the number of clusters (K)
--------------------------------------------------
   K = 2 clusters choisi
   - Cluster 0 : transactions normales
   - Cluster 1 : transactions frauduleuses
   Justification : classification binaire (fraude/normal)

   Train:      454904 échantillons
   Validation: 45490 échantillons
   Test:       113726 échantillons
   Train - Normal: 227452, Fraude: 227452
   Test  - Normal: 56863, Fraude: 56863

3. Initialize K clust